# 05, reading the misses

The failure taxonomy of the plan: 100 band misses drawn with the default seed from development, never test, read one by one and coded into categories named while reading (scripts/draw_misses.py draws them, results/models/taxonomy.json holds the codes). This notebook shows the misses as they were read and the counts the manifest carries.

In [1]:
import json

import numpy as np
import pandas as pd
import polars as pl

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_width_chars(110)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)
SEED = 20261114  # evaluation/phase2_plan.yaml, defaults.seed
print("numpy", np.__version__, "pandas", pd.__version__, "polars", pl.__version__)

numpy 2.5.4 pandas 2.3.3 polars 2.1.0


In [2]:
misses = json.load(open("results/models/misses.json"))
taxonomy = json.load(open("results/models/taxonomy.json"))
print(f"{misses['drawn']} drawn from {misses['development_misses']:,} development misses of "
      f"{misses['development_rows_with_band']:,} CVEs with an NVD band, model {misses['model']}")
frame = pd.DataFrame(misses["misses"])
frame["code"] = frame["cve_id"].map(taxonomy["codes"])
frame[["cve_id", "filer", "filer_band", "predicted_band", "nvd_band", "code"]].head(12)

100 drawn from 7,757 development misses of 23,272 CVEs with an NVD band, model fallback


            cve_id  ...                                                  code
0   CVE-2020-26303  ...                    The filer scored in CVSS 4.0 alone
1   CVE-2021-22281  ...                   The NVD rescored the filer's vector
2   CVE-2021-46990  ...      A kernel or hypervisor fix that states no impact
3   CVE-2021-47147  ...      A kernel or hypervisor fix that states no impact
4   CVE-2021-47539  ...      A kernel or hypervisor fix that states no impact
5   CVE-2022-48620  ...  A short text with no score, scored severe by the NVD
6   CVE-2022-48629  ...      A kernel or hypervisor fix that states no impact
7   CVE-2022-48643  ...      A kernel or hypervisor fix that states no impact
8   CVE-2022-48699  ...      A kernel or hypervisor fix that states no impact
9   CVE-2022-48774  ...      A kernel or hypervisor fix that states no impact
10  CVE-2022-48897  ...      A kernel or hypervisor fix that states no impact
11  CVE-2023-27335  ...                   The NVD rescored the f

In [3]:
order = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
step = frame.apply(lambda r: order.index(r["predicted_band"]) - order.index(r["nvd_band"]), axis=1)
values, counts = np.unique(np.abs(step), return_counts=True)
print("bands away from the NVD:", {int(v): int(n) for v, n in zip(values, counts)})
print("too high:", int((step > 0).sum()), "too low:", int((step < 0).sum()))

bands away from the NVD: {1: 91, 2: 9}
too high: 62 too low: 38


In [4]:
pd.DataFrame(taxonomy["categories"])[["name", "count", "meaning"]]

                                                   name  ...                                                                meaning
0      A kernel or hypervisor fix that states no impact  ...  a Linux or Xen commit message names the bug and its fix but not wh...
1                   The NVD rescored the filer's vector  ...  the filer gave a CVSS 3 vector and its band, the NVD changed one o...
2               A word's usual band, wrong for this CVE  ...  the model followed a word in the text, a weakness class or VulDB's...
3        The model left a band that agreed with the NVD  ...  the filer's band or the CISA ADP band matched the NVD's, and the m...
4  A short text with no score, scored severe by the NVD  ...  a sentence or two and no filer vector; the NVD assumed the worst c...
5                    The filer scored in CVSS 4.0 alone  ...  a CVSS 4.0 vector gives no CVSS 3 band or base metrics, so the mod...
6   The text states the impact and the model misread it  ...  the descriptio

Each category was named while reading, not before; the meaning column is the sentence written for it then, and the rule at the top of results/models/taxonomy.json says which category a miss takes when two fit. The two largest are out of the model's reach on these inputs: a Linux or Xen commit message says what was fixed and not what an attacker gains, so the NVD's band rests on a reading the text does not give, and a filer's vector that the NVD rescored leaves the model with the filer's band. The misses a better model could win back are the ones where it left a filer or CISA band that already agreed with the NVD, and the ones where a word in the text, often VulDB's own rating, pulled it to the band that word usually means. The counts are in the manifest as models.taxonomy, and the console shows them on /app/models.